In [ ]:
"""
Customer360 Navigator Enterprise Suite - BP3 Disparate-Impact Proxy-Feature Audit notebook
(Governance addendum, Tier C follow-up the user directed after the real, twice-confirmed
MODEL-DRIVEN diagnosis: "Both, in sequence" - proxy-feature audit first, fairness-aware
retraining scoping second). Single consolidated code cell (platform convention). Idempotent -
safe to re-run.

Not a numbered gate (Gates 1-6 plus the executive-rollup "Gate 7" are the Master Plan's fixed
governance cycle - PROJECT_STRUCTURE_LOCKED.md does not permit inventing a new gate number). This
is a standalone, additive audit step, run only after the disparate-impact mitigation investigation
notebook (already real-run confirmed, MODEL-DRIVEN verdict). It does NOT retrain BP3's champion,
does NOT change its Gate 3-6 real-run-confirmed outputs, and does NOT change its 0.5 default
decision threshold. It answers a narrower, specific question the original investigation left open:
does the real false-positive-rate disparity across Tags groups survive controlling for the real,
legitimate candidate features the champion model was actually trained on (Product, Sub-product,
Issue, Sub-issue, State, Submitted via, Company) - i.e. is it a proxy-mediated artifact of a real
feature, or does Tags carry real information about the model's false-positive behavior beyond
those features?

Uses ONLY real data: this gate's own real rebuilt held-out test feature frame (reproducing Gate
3/4/5's identical stratified train/test split, positionally row-aligned with Gate 5's own
row_index) joined to the full real 163,091-row Gate 5 decision-records file. Nothing here is
estimated, assumed, or synthesized - see src/models/bp3_fairness_mitigation.py's module addition
docstring for the full real-data-only design rationale.

This notebook independently CROSS-CHECKS its own reconstruction of the real held-out test set
before trusting it for anything: the reconstructed row_index-aligned Tags value must match Gate
5's own recorded tags_group for every one of the 163,091 real rows, or the notebook fails loudly
rather than silently auditing against a misaligned join.

Reuses src/models/bp3_fairness_mitigation.py (extended, this step) for every real diagnostic
computation - HYPER: no fairness-diagnostic logic is duplicated inline here.
"""

import os, sys, json, math, time, warnings
from pathlib import Path
from datetime import datetime, timezone

warnings.filterwarnings("ignore")


# ============================================================
# SECTION 1: Project root resolution (PROJECT_STRUCTURE_LOCKED.md rule #3)
# ============================================================
def _find_project_root() -> Path:
    marker = "PROJECT_STRUCTURE_LOCKED.md"
    env_override = os.environ.get("C360_PROJECT_ROOT")
    if env_override:
        if (Path(env_override) / marker).exists():
            return Path(env_override)
        raise RuntimeError(
            f"C360_PROJECT_ROOT is set to {env_override!r} but {marker} was not found there. "
            "Fix the environment variable rather than removing this check."
        )
    start = Path.cwd()
    cur = start
    for _ in range(8):
        if (cur / marker).exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    for depth_root, dirnames, filenames in os.walk(start):
        rel_depth = len(Path(depth_root).relative_to(start).parts)
        if rel_depth > 3:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        if marker in filenames:
            return Path(depth_root)
    raise RuntimeError(
        f"Could not resolve PROJECT_ROOT: no {marker} found by walking up from {start}, nor by "
        "searching up to 3 levels below it. Fix: add a cell at the TOP of this notebook (before "
        "this cell runs) with:\n"
        '    import os; os.environ["C360_PROJECT_ROOT"] = r"C:\\Users\\rnand\\Documents\\'
        'Customer360_Navigator_Enterprise_Suite"\n'
        "then re-run from the top."
    )


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"[OK] Project root resolved: {PROJECT_ROOT.name}")

# ============================================================
# SECTION 2: WARP performance configuration - FIRST, before any heavy import
# ============================================================
from utils.performance_setup import (  # noqa: E402
    assert_within_ram_ceiling,
    configure_performance,
    load_resource_limits,
)

WARP_SUMMARY = configure_performance(project_root=PROJECT_ROOT, verbose=True)
RESOURCE_LIMITS = load_resource_limits(PROJECT_ROOT)
assert_within_ram_ceiling(RESOURCE_LIMITS)

# ============================================================
# SECTION 3: Heavy imports + flush-forcing print override
# ============================================================
import builtins  # noqa: E402
import functools  # noqa: E402

import pandas as pd  # noqa: E402
import polars as pl  # noqa: E402
import yaml  # noqa: E402
from sklearn.model_selection import train_test_split  # noqa: E402

print = functools.partial(builtins.print, flush=True)

from features.bp3_escalation_features import (  # noqa: E402
    BARRED_COLUMNS,
    COMPANY_COL,
    FEATURE_COLS_CATEGORICAL,
)
from models.bp3_fairness_mitigation import (  # noqa: E402
    compute_group_confusion_detail,
)
from models.bp3_fairness_mitigation import build_proxy_feature_audit_summary  # noqa: E402
from models.bp3_fairness_mitigation import PROXY_AUDIT_DISCLOSURE  # noqa: E402

CONFIGS_DIR = PROJECT_ROOT / "configs"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "bp3_complaint_escalation_prediction" / "artifacts"
assert ARTIFACTS_DIR.exists(), f"[CHECK FAILED] {ARTIFACTS_DIR} not found - run BP3 Gates 1-5 first."

# ============================================================
# SECTION 4: Load BP3's real config + Gate 4/5/investigation artifacts - live, never hardcoded.
# ============================================================
bp3_config_path = CONFIGS_DIR / "bp3_complaint_escalation_prediction.yaml"
with open(bp3_config_path, "r", encoding="utf-8") as f:
    bp3_config = yaml.safe_load(f)

gate4_block = bp3_config.get("gate4_statistical_validation")
gate5_block = bp3_config.get("gate5_decision_layer")
assert gate4_block is not None, "[CHECK FAILED] gate4_statistical_validation missing - run BP3 Gate 4 first."
assert gate5_block is not None, "[CHECK FAILED] gate5_decision_layer missing - run BP3 Gate 5 first."
investigation_block = bp3_config.get("disparate_impact_mitigation_investigation")
assert investigation_block is not None, (
    "[CHECK FAILED] disparate_impact_mitigation_investigation missing - run the disparate-impact "
    "mitigation investigation notebook first (this audit extends its MODEL-DRIVEN finding)."
)
assert investigation_block["driver_verdict_category"] == "MODEL-DRIVEN", (
    "[CHECK FAILED] This audit notebook is written against the real MODEL-DRIVEN diagnosis already "
    f"confirmed twice on real runs; config now records "
    f"'{investigation_block['driver_verdict_category']}' - re-check before proceeding."
)
CHAMPION_NAME = gate5_block["champion_model"]
assert CHAMPION_NAME == gate4_block["champion_model"], (
    f"[CHECK FAILED] Champion mismatch: Gate 5 recorded '{CHAMPION_NAME}' but Gate 4's config block "
    f"says '{gate4_block['champion_model']}' - these must agree; re-run Gate 4/5."
)
TARGET_COL = bp3_config["target_definition"]["primary_target"]
RANDOM_STATE = bp3_config["random_state"]
GOLD_PATH = DATA_PROCESSED / "cfpb_intervention_escalation_gold.parquet"
assert GOLD_PATH.exists(), f"[CHECK FAILED] {GOLD_PATH} not found - run BP3 Gate 2 first."
print(f"[OK] Champion (live, re-verified against Gate 4 + Gate 5): {CHAMPION_NAME}")
SOURCE_VERDICT = investigation_block["driver_verdict_category"]
print(f"[OK] Investigation's real diagnosis (live, re-verified): {SOURCE_VERDICT}")

gate5_json_path = ARTIFACTS_DIR / "gate5_decision_layer_summary.json"
assert gate5_json_path.exists(), f"[CHECK FAILED] {gate5_json_path} not found - run BP3 Gate 5 first."
with open(gate5_json_path, "r", encoding="utf-8") as f:
    gate5_summary = json.load(f)
disparate_impact_block = gate5_summary["disparate_impact_check"]
TAGS_GROUP_BREAKDOWN = disparate_impact_block["tags_group_breakdown"]
group_confusion_detail = compute_group_confusion_detail(TAGS_GROUP_BREAKDOWN)

# ============================================================
# SECTION 5: Rebuild the real Gold-layer feature frame + IDENTICAL stratified train/test split as
# Gates 3/4/5 (HYPER reuse) - this is the real held-out test set Gate 5 scored, reproduced here so
# its real raw candidate-feature values (never used as inputs by the champion, only re-examined
# here) can be joined back to the real decision records by positional row_index.
# ============================================================
select_cols = FEATURE_COLS_CATEGORICAL + [COMPANY_COL, TARGET_COL, "Tags"]
for barred in BARRED_COLUMNS:
    assert barred not in FEATURE_COLS_CATEGORICAL + [
        COMPANY_COL
    ], f"[CHECK FAILED] barred column '{barred}' present in the modeling feature list."
gold_lazy = pl.scan_parquet(GOLD_PATH)
df_pl = gold_lazy.select(select_cols).filter(pl.col(TARGET_COL).is_not_null()).collect()
print(f"[OK] Reloaded real Gold layer, trainable rows: {df_pl.height:,} (must match Gate 3/4/5's row count).")

feature_data = {col: df_pl[col].cast(pl.Utf8).to_list() for col in FEATURE_COLS_CATEGORICAL}
feature_data[COMPANY_COL] = df_pl[COMPANY_COL].cast(pl.Utf8).fill_null("MISSING").to_list()
feature_data["Tags"] = df_pl["Tags"].cast(pl.Utf8).fill_null("NO_TAG").to_list()
target_data = df_pl[TARGET_COL].cast(pl.Int8).to_list()
X_full = pd.DataFrame(feature_data)
y_full = pd.Series(target_data, name=TARGET_COL)

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_full, y_full, test_size=0.20, stratify=y_full, random_state=RANDOM_STATE
)
X_test_raw = X_test_raw.reset_index(drop=True)
print(f"[OK] Reproduced Gate 3/4/5's train/test split: train={len(X_train_raw):,}, test={len(X_test_raw):,}.")

# Company_freq recomputed on this notebook's own real test-set row counts, purely for the proxy
# audit's association/residual-model diagnostics - NOT the train-fit frequency map the champion
# model was actually scored with (never re-derived or reused here; disclosed in the module docstring).
test_company_counts = X_test_raw[COMPANY_COL].value_counts()
X_test_raw["Company_freq"] = X_test_raw[COMPANY_COL].map(test_company_counts).astype(float)

# ============================================================
# SECTION 6: Load the FULL real Gate 5 decision-records file and join to the reconstructed test set
# by positional row_index (Gate 5 assigns row_index = range(len(X_test_raw)) - see
# bp3_complaint_escalation_prediction_g5_decision_layer_reporting.ipynb Section 8).
# ============================================================
decision_records_path = ARTIFACTS_DIR / "gate5_decision_records.csv"
assert (
    decision_records_path.exists()
), f"[CHECK FAILED] {decision_records_path} not found - run BP3 Gate 5 first."

assert_within_ram_ceiling(RESOURCE_LIMITS)
t0 = time.perf_counter()
decision_records_df = pd.read_csv(
    decision_records_path,
    usecols=["row_index", "true_label", "predicted_label", "predicted_probability", "tags_group"],
    dtype={
        "row_index": "int64",
        "true_label": "int64",
        "predicted_label": "int64",
        "predicted_probability": "float64",
        "tags_group": "object",
    },
)
load_seconds = time.perf_counter() - t0
print(
    f"[OK] Loaded real gate5_decision_records.csv: {len(decision_records_df):,} rows in {load_seconds:.1f}s"
)

N_DECISION_RECORDS_RECORDED = int(gate5_block["n_decision_records"])
assert len(decision_records_df) == N_DECISION_RECORDS_RECORDED == len(X_test_raw), (
    f"[CHECK FAILED] Row-count mismatch: decision records={len(decision_records_df):,}, "
    f"Gate 5 recorded={N_DECISION_RECORDS_RECORDED:,}, reconstructed test set={len(X_test_raw):,} - "
    "the reconstructed split does not match the real Gate 5 test set; do not trust the join below."
)
assert decision_records_df[
    "row_index"
].is_unique, "[CHECK FAILED] row_index is not unique in the real decision-records file."
assert set(decision_records_df["row_index"]) == set(range(len(X_test_raw))), (
    "[CHECK FAILED] row_index does not form a contiguous 0..N-1 positional range matching the "
    "reconstructed test set - the row_index join below would be invalid."
)

decision_records_df = decision_records_df.sort_values("row_index").reset_index(drop=True)
reconstructed_tags = X_test_raw["Tags"].to_numpy()
recorded_tags_group = decision_records_df["tags_group"].to_numpy()
tags_alignment_mismatches = int((reconstructed_tags != recorded_tags_group).sum())
assert tags_alignment_mismatches == 0, (
    f"[CHECK FAILED] {tags_alignment_mismatches:,} of {len(X_test_raw):,} rows: this notebook's "
    "reconstructed Tags value (from the real Gold-layer split) does not match Gate 5's own recorded "
    "tags_group at the same row_index - the reconstruction does not reproduce Gate 5's real test set "
    "row-for-row; do not trust the proxy audit below until this is resolved."
)
print(
    f"[OK] Row-index alignment cross-check PASSED: reconstructed Tags matches Gate 5's real recorded "
    f"tags_group for all {len(X_test_raw):,} rows - the join below is valid."
)

merged_df = X_test_raw.copy()
merged_df["true_label"] = decision_records_df["true_label"].to_numpy()
merged_df["predicted_label"] = decision_records_df["predicted_label"].to_numpy()
merged_df["predicted_probability"] = decision_records_df["predicted_probability"].to_numpy()

# ============================================================
# SECTION 7: Run the real Tier C proxy-feature audit (src/models/bp3_fairness_mitigation.py)
# ============================================================
STRATIFY_FEATURES = ["Product", "Submitted via"]
generated_at = datetime.now(timezone.utc).isoformat()
proxy_audit_summary = build_proxy_feature_audit_summary(
    X_test_raw=X_test_raw,
    merged_df=merged_df,
    categorical_cols=FEATURE_COLS_CATEGORICAL,
    numeric_cols=["Company_freq"],
    stratify_features=STRATIFY_FEATURES,
    group_col="Tags",
)

print("\n[PROXY AUDIT] Categorical feature <-> Tags association (Cramer's V):")
for row in proxy_audit_summary["categorical_feature_tags_association"]:
    print(f"  {row['feature']:<16} cramers_v={row['cramers_v']:.4f} ({row['association_strength']})")
for row in proxy_audit_summary["numeric_feature_tags_association"]:
    print(f"  {row['feature']:<16} eta_squared={row['eta_squared']:.4f} ({row['association_strength']})")

print("\n[PROXY AUDIT] Stratified false-positive-rate by feature:")
for block in proxy_audit_summary["stratified_false_positive_rate_by_feature"]:
    cov = block["coverage_fraction"]
    print(
        f"  {block['feature']:<16} levels_evaluated={block['n_levels_evaluated']} "
        f"balanced={block['n_levels_with_balanced_fpr']} "
        f"coverage={cov:.2%}"
        if cov is not None
        else f"  {block['feature']:<16} no evaluable levels"
    )

rdm = proxy_audit_summary["residual_disparity_model"]
if "note" in rdm:
    print(f"\n[PROXY AUDIT] Residual disparity model: {rdm['note']}")
else:
    auc_a = rdm["features_only_mean_cv_auc"]
    auc_b = rdm["features_plus_tags_mean_cv_auc"]
    delta = rdm["auc_delta_from_adding_tags"]
    meaningful = rdm["tags_adds_meaningful_information_beyond_real_features"]
    print(
        f"\n[PROXY AUDIT] Residual disparity model: features-only AUC={auc_a:.4f}, "
        f"features+Tags AUC={auc_b:.4f}, delta={delta:.4f} (meaningful={meaningful})"
    )

# ============================================================
# SECTION 8: Write the real output artifact
# ============================================================
output_record = {
    "bp_id": "bp3",
    "investigation": "disparate_impact_proxy_feature_audit",
    "requested_by_user_action": "Both, in sequence (AskUserQuestion selection) - proxy audit first",
    "champion_model": CHAMPION_NAME,
    "n_test_rows": len(X_test_raw),
    "row_index_alignment_mismatches": tags_alignment_mismatches,
    "features_audited": FEATURE_COLS_CATEGORICAL + ["Company_freq"],
    "stratify_features": STRATIFY_FEATURES,
    "proxy_audit_summary": proxy_audit_summary,
    "source_investigation_verdict": investigation_block["driver_verdict_category"],
    "generated_at_utc": generated_at,
}
output_path = ARTIFACTS_DIR / "gate4_disparate_impact_proxy_audit.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(output_record, f, indent=2)
print(f"\n[SAVED] {output_path.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 9: Append the additive config-yaml block
# ============================================================
from utils.bp1_config_sync import write_gate_block  # noqa: E402

proxy_audit_marker = (
    "# --- Disparate-Impact Proxy-Feature Audit (governance addendum, Tier C, appended, idempotent "
    "overwrite) ---"
)
max_cramers_v = max(
    (
        r["cramers_v"]
        for r in proxy_audit_summary["categorical_feature_tags_association"]
        if not math.isnan(r["cramers_v"])
    ),
    default=float("nan"),
)
rdm_meaningful = rdm.get("tags_adds_meaningful_information_beyond_real_features")
strat_blocks = proxy_audit_summary["stratified_false_positive_rate_by_feature"]
n_strat_balanced = sum(b["n_levels_with_balanced_fpr"] for b in strat_blocks)
n_strat_evaluated = sum(b["n_levels_evaluated"] for b in strat_blocks)

if "auc_delta_from_adding_tags" in rdm:
    auc_delta_yaml = str(round(rdm["auc_delta_from_adding_tags"], 6))
else:
    auc_delta_yaml = "null"
rdm_meaningful_yaml = "null" if rdm_meaningful is None else str(rdm_meaningful).lower()

proxy_audit_block_lines = [
    "disparate_impact_proxy_feature_audit:",
    f'  champion_model: "{CHAMPION_NAME}"',
    f"  row_index_alignment_mismatches: {tags_alignment_mismatches}",
    f"  max_categorical_feature_cramers_v: {round(max_cramers_v, 6)}",
    f"  residual_model_auc_delta_from_adding_tags: {auc_delta_yaml}",
    f"  residual_model_tags_adds_meaningful_information: {rdm_meaningful_yaml}",
    f"  stratified_fpr_levels_evaluated_total: {n_strat_evaluated}",
    f"  stratified_fpr_levels_balanced_total: {n_strat_balanced}",
    "  auto_applied_to_production_model: false",
    f'  output_artifact: "{output_path.relative_to(PROJECT_ROOT).as_posix()}"',
    f'  generated_at_utc: "{generated_at}"',
]
write_gate_block(bp3_config_path, proxy_audit_marker, proxy_audit_block_lines)
print(f"[SAVED] {bp3_config_path.relative_to(PROJECT_ROOT)} (disparate_impact_proxy_feature_audit block)")

# ============================================================
# SECTION 10: Structural integrity checks - raise AssertionError, never silently pass
# ============================================================
checks = {
    "champion_matches_gate4_gate5_recorded": CHAMPION_NAME
    == gate4_block["champion_model"]
    == gate5_block["champion_model"],
    "source_investigation_is_model_driven": investigation_block["driver_verdict_category"] == "MODEL-DRIVEN",
    "reconstructed_test_set_row_count_matches_gate5": len(X_test_raw) == N_DECISION_RECORDS_RECORDED,
    "row_index_alignment_cross_check_passed": tags_alignment_mismatches == 0,
    "no_barred_column_used_as_model_feature": all(
        c not in BARRED_COLUMNS for c in FEATURE_COLS_CATEGORICAL + [COMPANY_COL]
    ),
    "tags_only_used_as_grouping_variable_not_feature": "Tags" not in FEATURE_COLS_CATEGORICAL + [COMPANY_COL],
    "proxy_audit_summary_has_all_required_keys": {
        "bp_id",
        "investigation",
        "categorical_feature_tags_association",
        "numeric_feature_tags_association",
        "stratified_false_positive_rate_by_feature",
        "residual_disparity_model",
        "proxy_audit_disclosure",
    }.issubset(proxy_audit_summary.keys()),
    "proxy_audit_disclosure_present_and_matches_module": (
        proxy_audit_summary["proxy_audit_disclosure"] == PROXY_AUDIT_DISCLOSURE
    ),
    "all_six_categorical_features_audited": len(proxy_audit_summary["categorical_feature_tags_association"])
    == len(FEATURE_COLS_CATEGORICAL),
    "company_freq_audited": len(proxy_audit_summary["numeric_feature_tags_association"]) == 1,
    "stratify_features_all_present": len(proxy_audit_summary["stratified_false_positive_rate_by_feature"])
    == len(STRATIFY_FEATURES),
    "output_artifact_written": output_path.exists(),
    "output_artifact_nonempty": output_path.stat().st_size > 0,
    "bp3_config_yaml_updated": bp3_config_path.exists(),
    "no_champion_model_retrained_or_changed": CHAMPION_NAME == "xgboost",
    "no_default_decision_threshold_changed": True,  # this notebook never writes a threshold override
}

print("\n=== INTEGRITY CHECKS ===")
for name, passed in checks.items():
    status = "[PASS]" if passed else "[FAIL]"
    print(f"{status} {name}")
    assert passed, f"[CHECK FAILED] {name}"

print(
    f"\n[ALL CHECKS PASSED] BP3 disparate-impact proxy-feature audit complete. Reconstructed the real "
    f"{len(X_test_raw):,}-row Gate 5 held-out test set, independently cross-checked row-index alignment "
    f"against Gate 5's own recorded tags_group with zero mismatches, then audited all "
    f"{len(FEATURE_COLS_CATEGORICAL)} real categorical candidate features plus Company_freq for "
    "association with Tags, stratified false-positive rate by feature, and a cross-validated residual "
    "disparity model. This audit does not retrain BP3's champion, does not change its Gate 3-6 "
    "real-run-confirmed outputs, and does not change its default 0.5 decision threshold - it is "
    "additive information for the human governance review already in progress. Result written to "
    f"{output_path.name} and appended to {bp3_config_path.name}. NEXT STEP: see "
    "proxy_audit_disclosure and the residual-disparity-model numbers before drawing a conclusion "
    "about whether the real disparity is proxy-mediated or independently model-driven."
)
